# Cluster Architecture & kubeadm

## What's covered

- **The control plane in detail** — every component you've been calling by name (`etcd`, `kube-apiserver`, `kube-scheduler`, `kube-controller-manager`, `cloud-controller-manager`), what each one actually does, and how they talk to each other
- **The request flow through `kube-apiserver`** — authentication → authorisation → admission → write to `etcd`
- **The node components** — `kubelet`, `kube-proxy`, the container runtime
- **The plug-in interfaces** — CRI for the runtime, CNI for networking, CSI for storage; the cluster doesn't ship these, you do
- **Static pods** — pod manifests in `/etc/kubernetes/manifests`, run by the kubelet without involving the API server; how the control plane bootstraps itself
- **`kubeadm`** — the canonical tool for installing a cluster from scratch (and the CKA's chosen install path)
- **The cluster PKI** — what certificates exist, where they live, who signs what
- **etcd backup and restore** — `etcdctl snapshot save / restore`; the single most important operational skill the CKA tests
- **Cluster upgrade with `kubeadm`** — control plane first, kubelets second, version-skew rules

By the end of this notebook you should be able to draw the cluster on a whiteboard, install a new cluster from scratch in your head, back up and restore `etcd`, and walk through what happens when you `kubectl apply` — from your terminal all the way to the kubelet starting a container. Notebook 01 is the prerequisite — you've already met every component once. This is the chapter where each one earns its own paragraph.

## The control plane in one diagram

We sketched this in notebook 01. Here it is with every piece named.

```
                       Control plane (one or more nodes)
  +----------------------------------------------------------------------+
  |                                                                      |
  |   +---------+         +-------------------+                          |
  |   |  etcd   | <-----> |  kube-apiserver   | <----- kubectl,          |
  |   +---------+         +---------+---------+        every client      |
  |    cluster's source             ^                                    |
  |    of truth                     | watch                              |
  |                                 |                                    |
  |        +------------------------+----------+                         |
  |        |                                   |                         |
  |  +-----+--------+                +---------+------+                  |
  |  |  scheduler   |                |  controller-   |                  |
  |  +--------------+                |  manager       |                  |
  |   watches Pending pods,          +----------------+                  |
  |   binds to nodes                 (Deployment ctrl, ReplicaSet ctrl,  |
  |                                   Job ctrl, Node ctrl, Endpoints     |
  |  +--------------+                 ctrl, ServiceAccount ctrl ...      |
  |  | cloud-       |                 ~30 controllers)                   |
  |  | controller-  |                                                    |
  |  | manager      |                                                    |
  |  +--------------+                                                    |
  |   talks to the cloud provider                                        |
  +----------------------------------------------------------------------+

                              Worker nodes
  +----------------------------------------------------------------------+
  |  +---------+   +------------+   +-----------+   +-------------+      |
  |  | kubelet |   | kube-proxy |   | container |   |   CNI       |      |
  |  +----+----+   +------+-----+   | runtime   |   |  (network   |      |
  |       | CRI           |         | (containerd  |   agent)     |      |
  |       v               v         |  or CRI-O)|   +-------------+      |
  |  +----------------------------------+        |                       |
  |  |        your pods                 |        |                       |
  |  +----------------------------------+        |                       |
  +----------------------------------------------------------------------+
```

Rules of thumb to memorise:

- **Only the API server talks to `etcd`.** Every other component talks to the API server.
- **The API server is stateless.** It can be horizontally scaled; the state lives in `etcd`.
- **Controllers don't poll, they watch.** A long-lived HTTPS stream from the API server delivers events.
- **Nodes don't trust each other.** Everything authenticates to the API server, which is the single source of trust.

The rest of this chapter walks each box.

## `etcd` — the cluster's source of truth

**`etcd`** is a distributed key-value store. Every Kubernetes object you've created — every pod, deployment, configmap, secret, namespace, node — is a row in `etcd`. If `etcd` goes away, the cluster's memory goes with it.

Properties that matter:

- **Strongly consistent.** Built on the Raft consensus algorithm; reads always see the latest committed write.
- **Highly available with an odd-number cluster.** Production runs 3 or 5 `etcd` members; can tolerate (N−1)/2 failures.
- **Listens on `2379` (client) and `2380` (peer)**. Both ports require TLS in any non-trivial cluster.
- **Authenticates clients with certificates.** The API server is its primary client and authenticates with `apiserver-etcd-client.crt`.

### etcd is often a static pod

In a kubeadm cluster, `etcd` runs as a *static pod* on the control-plane nodes — defined by a manifest at `/etc/kubernetes/manifests/etcd.yaml`. The kubelet starts it before anything else.

The alternative — **external etcd** — runs `etcd` on dedicated machines outside the kubelet's control. More moving parts; better blast-radius isolation. Big managed offerings tend to use this layout.

### What's in `etcd`?

Every Kubernetes object, serialised as Protobuf (or JSON in older clusters), under a key like:

```
/registry/pods/default/web-7c9d-x2k5p
/registry/services/default/api
/registry/secrets/team-a/db-credentials
```

You can poke at it with `etcdctl`, but in practice you talk to the API server (`kubectl get`) and let it talk to `etcd`. Direct `etcd` reads are reserved for forensics and backups.

### Backup and restore — the CKA flag-bearer skill

We cover the actual commands later in this notebook. Internalise now: **`etcd` backup is the cluster's life insurance.** Lose `etcd`, lose the cluster — no objects, no PV bindings, no nothing. Backup is a `snapshot save`; restore is a `snapshot restore` followed by pointing `etcd` at the new data directory. The CKA reliably tests both.

## `kube-apiserver` — the front door

The API server is the only thing that ever writes to `etcd`. It's also the only thing every other component talks to. Killing it doesn't kill the workloads (they keep running where they are), but it stops the cluster from making any new decisions.

### The request flow

Every API request — `kubectl get pods`, the scheduler watching for new pods, the kubelet patching pod status, your CI pipeline applying a manifest — goes through the same five stages:

```
  +-------+    +-------+    +-----------+    +------------+    +------+
  | AUTHN | -> | AUTHZ | -> | ADMISSION | -> | VALIDATION | -> | etcd |
  | who?  |    | can?  |    | mutate +  |    | schema     |    |      |
  |       |    | (RBAC)|    | validate  |    | check      |    |      |
  +-------+    +-------+    +-----------+    +------------+    +------+
```

**1. Authentication (AUTHN).** *Who is this request from?* The API server tries each configured authenticator in order — client certificates (`/etc/kubernetes/pki/`), bearer tokens (ServiceAccount JWTs, static tokens), OpenID Connect, webhook. The first one that recognises the credential wins; the request is now tagged with the user identity. If none recognises, 401.

**2. Authorisation (AUTHZ).** *Can this user do this action on this object?* Usually **RBAC** (notebook 10): match the verb (`get`, `list`, `create`, `update`, `delete`, `watch`, `patch`, ...) against the user's bound roles. Can be combined with Node authorisation (kubelets only get what they need), webhook authorisation, ABAC (deprecated). If denied, 403.

**3. Admission control.** The request is *allowed* — but should it be *modified*, or rejected for policy reasons?

- **Mutating admission controllers** can change the request. Examples: `MutatingAdmissionWebhook`, the `ServiceAccount` controller (mounts a token), `LimitRanger` (fills in default requests).
- **Validating admission controllers** can reject the request. `ResourceQuota`, `PodSecurity`, custom validating webhooks (Gatekeeper, Kyverno, company policies).

**4. Validation.** Final schema check against the object kind. Required fields present? Field types right? Immutable fields not changed? If not, 422.

**5. Persistence.** The (possibly mutated) object is written to `etcd`. After this, watchers see the change and the controllers do their thing.

### Why this matters

Most "I applied this and nothing happened" debugging traces back to one of these stages:

- 401 → AUTHN — bad credential, expired cert.
- 403 → AUTHZ — RBAC rule missing.
- Webhook error → admission — some operator's validating webhook rejected you.
- 422 → validation — the field doesn't exist on that kind / version.

`kubectl --v=8 apply ...` shows the full HTTP traffic, including which stage rejected what. Worth knowing for the exam's troubleshooting bucket (notebook 10).

### The API server's flags

The API server takes a long list of flags. The ones the CKA expects you to recognise:

- `--etcd-servers=https://127.0.0.1:2379` — where `etcd` lives.
- `--service-cluster-ip-range=10.96.0.0/12` — the Service IP pool.
- `--authorization-mode=Node,RBAC` — which authorisers, in order.
- `--enable-admission-plugins=NamespaceLifecycle,LimitRanger,ServiceAccount,...` — admission chain.
- `--client-ca-file`, `--tls-cert-file`, `--tls-private-key-file` — TLS material.

You'll see all of these in `/etc/kubernetes/manifests/kube-apiserver.yaml` on a kubeadm node.

## `kube-scheduler` and `kube-controller-manager`

Two control-plane workhorses you've already met.

### `kube-scheduler` (recap from notebook 07)

Watches `Pending` pods, filters and scores nodes, binds the pod to the winning node. Pluggable via scheduling profiles. Runs as a static pod from `/etc/kubernetes/manifests/kube-scheduler.yaml`. Talks to the API server, never to nodes directly.

### `kube-controller-manager`

A single binary that hosts roughly **30 distinct controllers**, each running the same observe-compare-act loop you've seen throughout this course. The ones that matter most:

| Controller | What it watches | What it does |
|---|---|---|
| **Node controller** | `Node` objects | Marks nodes `NotReady` when they stop heart-beating; manages `NoExecute` taints. |
| **Deployment** | `Deployment` | Creates and manages `ReplicaSet`s for rollouts (notebook 03). |
| **ReplicaSet** | `ReplicaSet` | Creates and deletes pods to match the desired count. |
| **DaemonSet, StatefulSet, Job, CronJob** | respective kinds | Create and manage their pods. |
| **Endpoints / EndpointSlice** | `Service` + Pods | Maintains the list of ready pod IPs behind each Service (notebook 04). |
| **ServiceAccount** | `Namespace` + `ServiceAccount` | Creates a default SA per namespace, and a token Secret per SA (legacy path). |
| **Namespace** | `Namespace` | Cascades deletes — when a namespace is `Terminating`, deletes everything in it. |
| **PV / PVC binder** | `PV`, `PVC` | Matches PVCs to PVs (notebook 06). |
| **HPA** | `HorizontalPodAutoscaler` | Adjusts Deployment replicas based on metrics. |
| **TTL, garbage collector** | objects with TTL or finalisers | Cleans up. |

Bundling them in one binary is just packaging — each controller logically runs on its own. They elect a leader (so only one instance is active at a time, even when the binary runs HA across multiple control-plane nodes) using lease objects in `kube-system`.

### `cloud-controller-manager`

The cloud-specific bits used to live in `kube-controller-manager`. They were extracted to a separate binary so non-cloud (and bare-metal) clusters don't need them, and so cloud providers can ship their own. It runs:

- **Node controller (cloud-side)** — talks to the cloud API to learn about VMs; adds and removes node addresses.
- **Route controller** — programs cloud routing for pod CIDRs.
- **Service controller** — provisions cloud load balancers for `Service` of type `LoadBalancer` (notebook 04).

In a cluster with no cloud (your `kind`), `cloud-controller-manager` isn't running at all.

## Node components — `kubelet`, `kube-proxy`, the container runtime

Every node, control-plane or worker, runs three things in coordination.

### `kubelet`

The agent on the node. **The kubelet doesn't run as a pod** — it's a system service started by `systemd` (or whatever init system the node uses). Its job:

1. Register the node with the API server.
2. Watch the API server for pods whose `spec.nodeName == <this node>`.
3. For each such pod, drive the container runtime to make it real: pull image, create network namespace, mount volumes, start containers.
4. Run **probes** (liveness, readiness, startup — notebook 02), restart containers per `restartPolicy`.
5. Report status back to the API server — phases, conditions, container states.
6. Monitor node-level pressure (memory, disk, PIDs) and **evict** when over thresholds.

The kubelet does **not** schedule. It only runs what's been bound to its node.

### `kube-proxy`

The data-plane half of Services. Watches Services and EndpointSlices, programs the node's kernel networking (`iptables` / `IPVS` / `nftables` — notebook 04) so the Service ClusterIPs route to real pod IPs. Runs as a DaemonSet pod in `kube-system`.

Optional in modern clusters — some CNI plugins replace it entirely (Cilium with eBPF, for example).

### Container runtime

What actually runs containers. Kubernetes doesn't ship a runtime; it speaks **CRI** (next section) to whichever one is installed. The two production-grade options:

- **`containerd`** — the most common choice today. Lightweight, from the Docker project; donated to CNCF.
- **`CRI-O`** — a Kubernetes-native runtime from Red Hat / the OpenShift ecosystem.

Docker Engine itself is no longer a supported runtime — the `dockershim` adapter that bridged kubelet to Docker was removed in Kubernetes 1.24. The runtime usually lives at a Unix socket like `/run/containerd/containerd.sock`; the kubelet's `--container-runtime-endpoint` flag points at it.

## The plug-in interfaces — CRI, CNI, CSI

Kubernetes is small at the core. Three big plug-in points let it talk to whatever runtime, network, and storage you have.

### CRI — Container Runtime Interface

A gRPC API the kubelet speaks to a local runtime. Two services:

- **`RuntimeService`** — creating pods and containers; lifecycle.
- **`ImageService`** — pulling and listing images.

The kubelet doesn't know whether `containerd` or `CRI-O` is on the other end of the socket; it just calls CRI. **One CRI implementation per node.**

### CNI — Container Network Interface

A spec (not gRPC; a small JSON-over-stdin protocol) for "set up a container's network namespace." When the kubelet creates a pod's network namespace, it invokes the configured CNI plugin, which:

1. Allocates a pod IP from the node's pool.
2. Programs routing so the pod can talk to other pods, Services, and out to the internet (subject to NetworkPolicy).
3. Hands back the IP for the kubelet to record in pod status.

Common CNI plugins: **Calico**, **Cilium**, **Flannel**, **Weave Net**, **Antrea**, **Amazon VPC CNI**, **Azure CNI**. They differ in:

- **Encapsulation** — VXLAN / IP-in-IP overlay vs flat L3 routing vs cloud-native (eBPF, AWS ENIs).
- **NetworkPolicy implementation** — iptables, IPVS, eBPF.
- **Performance** — eBPF-based plugins (Cilium) can skip iptables entirely.

`kind` ships with `kindnet`. Multi-CNI options ("Multus") exist for advanced cases.

### CSI — Container Storage Interface

Covered in notebook 06. gRPC API between Kubernetes and storage backends. One CSI driver per backend you want to talk to (EBS, GCE PD, NFS, Ceph, local-path).

### The point of all three

Kubernetes' job is *orchestration* — placing workloads, managing their lifecycle, handling failures. The actual *doing* — running containers, plumbing networks, attaching disks — is delegated. Same orchestration logic runs on a Raspberry Pi (k3s, containerd, flannel, local storage) or on a 1000-node cloud cluster (EKS, containerd, AWS VPC CNI, EBS CSI).

## Static pods — how the control plane bootstraps itself

A **static pod** is a pod the kubelet runs directly from a manifest on disk, without involving the API server at all.

The kubelet watches a directory — by default `/etc/kubernetes/manifests` — for YAML files. Each file is a pod manifest. The kubelet:

1. Reads the file.
2. Starts the pod.
3. Creates a *mirror pod* in the API server so `kubectl get pods` can see it (named `<pod>-<node>`).
4. Keeps the pod alive (restarts on crash) and watches the file for changes.

**You can't `kubectl delete` a static pod.** Deleting the mirror pod just removes it from the API server's view; the kubelet will recreate the mirror within seconds. To stop a static pod, **move or delete its manifest file** on the node.

### Why this exists

The control plane has a chicken-and-egg problem: how do you start the API server when you need the API server to start it? Static pods are the answer.

On a kubeadm-installed control-plane node, `/etc/kubernetes/manifests/` contains:

```
/etc/kubernetes/manifests/etcd.yaml
/etc/kubernetes/manifests/kube-apiserver.yaml
/etc/kubernetes/manifests/kube-controller-manager.yaml
/etc/kubernetes/manifests/kube-scheduler.yaml
```

The kubelet starts on system boot, reads those files, and brings up the control plane. **`etcd` first** (the API server needs it), then the API server, then the rest.

### Static pods are useful outside the control plane too

For node-level agents that *must* run even if the API server is unreachable — a node-monitoring daemon, an emergency log shipper — a static pod is the right answer. Don't bother for application workloads; use a DaemonSet (which the kubelet runs *via* the API server).

### Reading the manifests

```bash
# On a control-plane node
ls /etc/kubernetes/manifests/
cat /etc/kubernetes/manifests/kube-apiserver.yaml
```

The flags you saw in the API server section are all there — long lists of `command:` args in the container spec.

In [ ]:
# Mirror pods for the control plane components show up here
!kubectl get pods -n kube-system
!echo '---'
# The static-pod components by node
!kubectl get pods -n kube-system -o custom-columns=NAME:.metadata.name,NODE:.spec.nodeName,STATUS:.status.phase | head -15
!echo '---'
# Peek at the static-pod manifests on the kind node (kind nodes are Docker containers)
!NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}') && \
  echo "node: $NODE" && \
  docker exec "$NODE" ls /etc/kubernetes/manifests/
!echo '---'
# A glimpse of the apiserver manifest — the long list of flags you'd otherwise pass to systemd
!NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}') && \
  docker exec "$NODE" head -40 /etc/kubernetes/manifests/kube-apiserver.yaml

## `kubeadm` — installing a cluster from scratch

`kubeadm` is the canonical installer for Kubernetes clusters. It doesn't provision machines, install the OS, configure firewalls, or pick a CNI — those are your problem. What it *does* do is set up the control plane and bootstrap join tokens, given that you have:

- A set of Linux machines with a container runtime installed.
- The `kubelet`, `kubeadm`, and `kubectl` binaries installed on each.
- A pod CIDR you've chosen.
- Network connectivity between nodes.

A typical install is **two commands**: `kubeadm init` on the first control-plane node, then `kubeadm join` on every other node.

### `kubeadm init` — bring up the control plane

On the chosen first control-plane node:

```bash
kubeadm init \
  --pod-network-cidr=192.168.0.0/16 \
  --apiserver-advertise-address=<this-node-ip> \
  --upload-certs                                 # for adding more control-plane nodes later
```

What happens, in order:

1. **Pre-flight checks.** Kernel version, swap off, ports free, runtime ready, group memberships. Fails loudly if anything's wrong.
2. **Generate the cluster PKI.** `kubeadm` creates a CA in `/etc/kubernetes/pki/` and signs certificates for the API server, etcd, kubelet client, and others.
3. **Write static-pod manifests.** `etcd.yaml`, `kube-apiserver.yaml`, `kube-controller-manager.yaml`, `kube-scheduler.yaml` go to `/etc/kubernetes/manifests/`.
4. **Wait for the control plane to come up.** The kubelet is already running (you started it before `kubeadm`); it reads the manifests and starts the static pods.
5. **Bootstrap addons.** Install CoreDNS as a Deployment; install `kube-proxy` as a DaemonSet.
6. **Print a `kubeadm join` command** with a token. Save this — you need it to add more nodes.

### `kubeadm join` — add a node

On each additional node (worker or control-plane):

```bash
# Worker
kubeadm join <control-plane-ip>:6443 \
  --token <token> \
  --discovery-token-ca-cert-hash sha256:<hash>

# Additional control-plane
kubeadm join <control-plane-ip>:6443 \
  --token <token> \
  --discovery-token-ca-cert-hash sha256:<hash> \
  --control-plane \
  --certificate-key <key from --upload-certs>
```

What happens:

1. The kubelet contacts the API server using the join token (bootstrap auth).
2. It receives the cluster's CA certificate and verifies it against the SHA-256 hash you passed in.
3. The Node is registered.
4. For a worker — the kubelet starts watching for pods bound to itself.
5. For a control-plane node — fetches the encrypted control-plane certs from a Secret in `kube-public`, decrypts with `--certificate-key`, writes its own static-pod manifests.

### Tokens are short-lived

`kubeadm init` gives you a 24-hour token by default. After it expires:

```bash
kubeadm token create --print-join-command         # generates a new join command
kubeadm token list                                # see active tokens
```

### Installing a CNI is your job

`kubeadm init` does *not* install a CNI. Until you do, all pods stay in `Pending` / `ContainerCreating` — they have no network. After init:

```bash
# Pick a CNI. Calico is the common choice; others work the same way.
kubectl apply -f https://raw.githubusercontent.com/projectcalico/calico/v3.27.0/manifests/calico.yaml
```

CoreDNS pods will start running once the CNI is up.

## The cluster PKI — who has what certificate

A Kubernetes cluster is held together by TLS. Almost every component talks to almost every other component over HTTPS, and identities are proven with X.509 client certificates.

### The CAs

A kubeadm cluster has *two* certificate authorities by default:

- **Kubernetes CA** (`/etc/kubernetes/pki/ca.crt`) — signs all the front-end certs (API server, kubelet clients, etc.).
- **etcd CA** (`/etc/kubernetes/pki/etcd/ca.crt`) — signs etcd's own server and peer certs.

A third **`front-proxy-ca`** is used by the API aggregation layer for extension API servers (metrics-server, custom resources). Less day-to-day relevance.

### Certificates by role

Living in `/etc/kubernetes/pki/`:

| Cert | Signed by | Identity | Used by |
|---|---|---|---|
| `apiserver.crt` | k8s CA | API server's hostnames | API server (TLS server) |
| `apiserver-kubelet-client.crt` | k8s CA | `kube-apiserver-kubelet-client` | API server → kubelet |
| `apiserver-etcd-client.crt` | etcd CA | API server | API server → etcd |
| `etcd/server.crt` | etcd CA | etcd's hostnames | etcd (TLS server) |
| `etcd/peer.crt` | etcd CA | etcd's hostnames | etcd ↔ etcd peers |
| `front-proxy-client.crt` | front-proxy CA | aggregation layer | API server → extension APIs |

Plus, per node, the kubelet has its own certificate at `/var/lib/kubelet/pki/kubelet-client-current.pem`. Modern clusters use the **kubelet certificate-rotation** mechanism — the kubelet asks the API server for a new cert before the old one expires.

### kubeconfig files

The control-plane components don't read certs directly; they read **kubeconfig** files that bundle a cert, key, and API server URL:

```
/etc/kubernetes/admin.conf            # for cluster admins — give to a human, never commit
/etc/kubernetes/kubelet.conf          # for the local kubelet (often replaced by rotated certs)
/etc/kubernetes/controller-manager.conf
/etc/kubernetes/scheduler.conf
```

Your laptop's `~/.kube/config` is the same format. `kubeadm` produces `admin.conf` and tells you to copy it to your home directory.

### Certificate expiry

The certs `kubeadm` issues have a **1-year lifetime** by default. Two ways to renew before they expire:

```bash
kubeadm certs check-expiration          # show what's expiring when
kubeadm certs renew all                 # renew everything; needs a control-plane restart
```

`kubeadm upgrade` (later in this notebook) also rotates certs as a side effect. **The CKA loves this question** — "your cluster stopped working a year after install; what's wrong?" Answer: certs expired.

In [ ]:
# All the PKI files on the kind control-plane node
!NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}') && \
  docker exec "$NODE" ls /etc/kubernetes/pki/
!echo '---'
# Inspect the API server's cert — see who it's for and when it expires
!NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}') && \
  docker exec "$NODE" openssl x509 -in /etc/kubernetes/pki/apiserver.crt -noout -subject -issuer -dates 2>/dev/null
!echo '---'
# Subject Alternative Names — who the cert claims to be
!NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}') && \
  docker exec "$NODE" openssl x509 -in /etc/kubernetes/pki/apiserver.crt -noout -ext subjectAltName 2>/dev/null
!echo '---'
# What kubeadm thinks about expiry across all the certs
!NODE=$(kubectl get nodes -o jsonpath='{.items[0].metadata.name}') && \
  docker exec "$NODE" kubeadm certs check-expiration 2>/dev/null | head -20

## etcd backup and restore — the CKA's flag-bearer skill

Lose `etcd`, lose the cluster. The CKA tests `etcd snapshot save / restore` reliably; learn it cold.

### Backup — `etcdctl snapshot save`

`etcdctl` is the CLI client for `etcd`. On a kubeadm cluster you can `kubectl exec` into the `etcd` static pod and run it there (the binary is bundled in the etcd image), or run it directly on the control-plane node if `etcdctl` is on the host's PATH. The required pieces of authentication:

- **`--endpoints=https://127.0.0.1:2379`** — where etcd is listening.
- **`--cacert=/etc/kubernetes/pki/etcd/ca.crt`** — etcd's CA.
- **`--cert=/etc/kubernetes/pki/etcd/server.crt`** — a client cert that etcd trusts.
- **`--key=/etc/kubernetes/pki/etcd/server.key`** — its private key.

Then the command itself:

```bash
ETCDCTL_API=3 etcdctl snapshot save /var/lib/etcd-backup/snap.db \
  --endpoints=https://127.0.0.1:2379 \
  --cacert=/etc/kubernetes/pki/etcd/ca.crt \
  --cert=/etc/kubernetes/pki/etcd/server.crt \
  --key=/etc/kubernetes/pki/etcd/server.key
```

A snapshot is a single file. Move it off the node — that's your backup.

### Verify a snapshot

```bash
ETCDCTL_API=3 etcdctl --write-out=table snapshot status /var/lib/etcd-backup/snap.db
```

Shows the snapshot's revision, the number of keys, and the storage hash. If `etcdctl` won't read the file, the file is broken — make another one.

### Restore — `etcdctl snapshot restore`

The standard CKA scenario: "the cluster is broken, here's a snapshot, restore it." Procedure:

1. **Stop the control plane.** On the node, move the static-pod manifests out of `/etc/kubernetes/manifests/` (for example, into `/tmp/`). The kubelet sees they're gone and stops the static pods. Wait until `etcd` and the API server are down.

2. **Restore the snapshot to a new data directory.** Do *not* overwrite the existing `/var/lib/etcd`.

   ```bash
   ETCDCTL_API=3 etcdctl snapshot restore /backup/snap.db \
     --data-dir=/var/lib/etcd-from-backup
   ```

3. **Point etcd at the new data directory.** In `/etc/kubernetes/manifests/etcd.yaml`, change the `volumes.hostPath.path` for the `etcd-data` volume from `/var/lib/etcd` to `/var/lib/etcd-from-backup`. (Or reuse the same path if you're confident — but a new directory is the safer pattern under exam pressure.)

4. **Restore the static-pod manifests.** Move them back into `/etc/kubernetes/manifests/`. The kubelet starts everything; `etcd` boots from the snapshot.

5. **Verify.** `kubectl get nodes`, `kubectl get pods -A`. The cluster should look like it did at the time of the backup.

### Things to memorise

- **`ETCDCTL_API=3`** — version 3 is the only one anyone uses; the env var is required on older `etcdctl` versions. Modern `etcdctl` (3.5+) defaults to v3.
- **Three certs.** ca, cert, key. They live under `/etc/kubernetes/pki/etcd/`.
- **A snapshot is point-in-time.** Anything created or changed after the snapshot is lost on restore.
- **External etcd is the same recipe, different host.** `--endpoints` points at the external cluster; certs come from wherever you stashed them.

In [ ]:
# Find the etcd static pod
!kubectl get pods -n kube-system -l component=etcd
!echo '---'
# Take a snapshot from inside the etcd container — etcdctl is on PATH there.
# kubeadm's etcd image (3.5+) defaults to API v3, so ETCDCTL_API is no longer needed.
!ETCD_POD=$(kubectl get pods -n kube-system -l component=etcd -o jsonpath='{.items[0].metadata.name}') && \
  echo "etcd pod: $ETCD_POD" && \
  kubectl exec -n kube-system "$ETCD_POD" -- etcdctl snapshot save /tmp/etcd-snap.db \
    --endpoints=https://127.0.0.1:2379 \
    --cacert=/etc/kubernetes/pki/etcd/ca.crt \
    --cert=/etc/kubernetes/pki/etcd/server.crt \
    --key=/etc/kubernetes/pki/etcd/server.key
!echo '---'
# Verify the snapshot — revision, key count, hash
!ETCD_POD=$(kubectl get pods -n kube-system -l component=etcd -o jsonpath='{.items[0].metadata.name}') && \
  kubectl exec -n kube-system "$ETCD_POD" -- etcdctl --write-out=table snapshot status /tmp/etcd-snap.db

## Cluster upgrade with `kubeadm`

You don't `kubectl` your way to a new Kubernetes version. You run `kubeadm upgrade` on the nodes themselves. The CKA may not ask you to upgrade end-to-end, but it will ask you to *describe* the order and the version-skew rules.

### The version-skew rule

Kubernetes' compatibility guarantees:

- **`kubectl`** — can be ±1 minor version from the API server.
- **`kube-apiserver`** — the *highest* version in the cluster. Everything else trails or matches.
- **`controller-manager`, `scheduler`, `cloud-controller-manager`** — same minor as the API server, or one minor older.
- **`kubelet`** — up to **two** minor versions older than the API server (Kubernetes 1.28+; previously one).
- **`kube-proxy`** — same minor as the kubelet on the node.

So a 1.29 API server can talk to 1.28 and 1.27 kubelets. A 1.27 API server **cannot** talk to a 1.29 kubelet — upgrade the control plane first, kubelets after.

### Upgrade order

For a multi-control-plane cluster:

1. **First control-plane node:**

   ```bash
   # Upgrade the kubeadm binary first (package manager)
   apt-get install -y kubeadm=1.29.x-...
   # Plan and apply
   kubeadm upgrade plan
   kubeadm upgrade apply v1.29.x
   # Drain the node, upgrade kubelet, uncordon
   kubectl drain <this-node> --ignore-daemonsets
   apt-get install -y kubelet=1.29.x-... kubectl=1.29.x-...
   systemctl daemon-reload && systemctl restart kubelet
   kubectl uncordon <this-node>
   ```

2. **Each additional control-plane node:** same, but use `kubeadm upgrade node` instead of `kubeadm upgrade apply`. The first node already did the cluster-wide work.

3. **Each worker node:** drain, upgrade `kubeadm` + `kubelet` + `kubectl`, restart kubelet, uncordon.

### What `kubeadm upgrade` actually does

- **Pulls new control-plane images** for the target version.
- **Rotates certificates** (a free renewal — useful even outside a version bump).
- **Updates static-pod manifests** in `/etc/kubernetes/manifests/` with the new image tags. The kubelet sees the change and restarts the static pods.
- **Updates kube-proxy and CoreDNS** addons.

The kubelet itself is *not* upgraded by `kubeadm upgrade`; you upgrade its package separately.

### Skip-level upgrades aren't allowed

You can only upgrade **one minor version at a time** — 1.27 → 1.28 → 1.29, never 1.27 → 1.29 in one shot. Patch versions are free (1.28.3 → 1.28.7 any time).

## What to practise

This chapter doesn't have its own cleanup — we didn't deploy any workloads. But it's the densest content in the course, and the CKA tests it heavily. Concrete things to practise before the exam:

- **Backup and restore `etcd`** on your own kind cluster. Time yourself; aim for under 5 minutes. Bring the cluster down, restore from a snapshot, bring it back up.
- **Find a control-plane component's flag** in the static-pod manifest. Pick one, change it, watch the kubelet restart the pod.
- **Read `kubeadm certs check-expiration`** output. Know which cert means what.
- **Walk through `kubeadm init` and `kubeadm join`** on paper. What happens at each step, in order.
- **Write the version-skew rule** from memory. Which component can lag, by how much.
- **Trace a `kubectl apply`** through AUTHN → AUTHZ → admission → validation → etcd. Know which response code each stage returns when it rejects.

Nothing to delete — the cluster's still where you left it. Notebook 09 is networking, ingress, and network policies; it builds on the Services chapter (04) and on this notebook's CNI and `kube-proxy` overview.